# 06 — Reproducción limpia del champion

Notebook independiente: no entrena nada. Carga `models:/qsrr-predict-modelo@champion` desde el Model Registry, predice el mismo holdout agrupado y comprueba que el RMSE es **exactamente** el registrado en la corrida del modelo.

In [1]:
from pathlib import Path
import os
import sys
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import mlflow
import mlflow.sklearn
import numpy as np
from mlflow.tracking import MlflowClient
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from src.datos import load_snapshot
from src.experiments import configure_tracking
from src.pipeline import model_inputs, valid_smiles_mask
from src.splits import create_or_load_holdout

MODEL_URI = "models:/qsrr-predict-modelo@champion"
configure_tracking()
client = MlflowClient()
version = client.get_model_version_by_alias("qsrr-predict-modelo", "champion")
logged = client.get_run(version.run_id).data.metrics
print({"version": version.version, "run_id": version.run_id, "holdout_rmse_registrado": logged["holdout_rmse"], "holdout_mae_registrado": logged["holdout_mae"]})

{'version': '5', 'run_id': '04483473d3314b51b7ee40f03ca90c94', 'holdout_rmse_registrado': 2.328971943011868, 'holdout_mae_registrado': 1.529281301169671}


In [2]:
frame, metadata = load_snapshot()
all_data = frame.loc[valid_smiles_mask(frame) & frame["inchikey"].notna()].reset_index(drop=True)
_, test, split_metadata = create_or_load_holdout(all_data, source_sha256=metadata["sha256"])
X_test, y_test, _ = model_inputs(test)
model = mlflow.sklearn.load_model(MODEL_URI)
prediction = model.predict(X_test)
reproduced = {
    "holdout_rmse": float(root_mean_squared_error(y_test, prediction)),
    "holdout_mae": float(mean_absolute_error(y_test, prediction)),
}
print({"filas_holdout": len(y_test), "sha256_snapshot": metadata["sha256"], **reproduced})
for name, value in reproduced.items():
    assert np.isclose(value, logged[name], rtol=0, atol=1e-12), (name, value, logged[name])
print("Reproducción exacta: las métricas coinciden con las registradas en MLflow.")

{'filas_holdout': 30216, 'sha256_snapshot': '0c366b465f7b56e6b5873132279f7a3152532f0a035186c776f491c01a628210', 'holdout_rmse': 2.328971943011868, 'holdout_mae': 1.529281301169671}
Reproducción exacta: las métricas coinciden con las registradas en MLflow.
